# 准备数据

使用孙衍刚的数据训练一下模型

In [1]:
%reset -f
import scanpy as sc

# 4m 5.3s
sc_adata = sc.read("mouseBrain_RNA_counts.h5ad")
sp_adata = sc.read("/media/williamhan/4E44841544840247/Python/bioinformatics_project/MCAO20250929/data/cellbin/MCAO20250929_reduce.h5ad")
sc_adata, sp_adata

(AnnData object with n_obs × n_vars = 378287 × 52198
     obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'libId', 'percent.mt', 'percent.ribo', 'countFeatureRatio', 'sampleId', 'batchId', 'maxPredClass', 'sampleRegion1', 'sampleRegion2', 'sampleRegion3', 'Cell_cluster', 'Cell_group', 'Description', 'Cell_subclass', 'Cell_class', 'Color', 'ident'
     uns: 'X_name'
     obsm: 'HARMONY', 'PCA', 'UMAP'
     layers: 'logcounts',
 AnnData object with n_obs × n_vars = 497563 × 19781
     obs: 'library', 'group', 'sample', 'accession', 'condition', 'time', 'batch', 'dnbCount', 'x', 'y', 'area', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'passing_mt', 'passing_ngenes', '_scvi_batch', '_scvi_labels'
     var: 'mt', 'ribo', 'hb', 'n_cel

In [ ]:
for key in ["logcounts"]:
    if key in sc_adata.layers:
        del sc_adata.layers[key]

# uns 和 obsm 是整个容器，不建议直接 del sc_adata.uns / del sc_adata.obsm
# 更稳妥是清空里面的内容
sc_adata.uns.clear()
sc_adata.obsm.clear()

# celltype 必须是 categorical
sc_adata.obs["celltype"] = sc_adata.obs["Cell_group"].astype("category")
# 必须有一列是 celltype
sc_adata.obs = sc_adata.obs.loc[:, ["celltype"]]
sc_adata

AnnData object with n_obs × n_vars = 378287 × 52198
    obs: 'celltype'

In [3]:
import numpy as np

# 1. 确保基因名唯一
sp_adata.var_names_make_unique()
sc_adata.var_names_make_unique()

# 2. 取交集
common_genes = sp_adata.var_names.intersection(sc_adata.var_names)

print(f"sp_adata genes: {sp_adata.n_vars}")
print(f"sc_adata genes: {sc_adata.n_vars}")
print(f"common genes: {len(common_genes)}")

# 3. 按同一组基因、同一顺序切片
sp_adata = sp_adata[:, common_genes].copy()
sc_adata = sc_adata[:, common_genes].copy()

# 4. 确认顺序一致
assert np.array_equal(sp_adata.var_names, sc_adata.var_names)

sc_adata, sp_adata

sp_adata genes: 19781
sc_adata genes: 52198
common genes: 19081


(AnnData object with n_obs × n_vars = 378287 × 19081
     obs: 'celltype',
 AnnData object with n_obs × n_vars = 497563 × 19081
     obs: 'library', 'group', 'sample', 'accession', 'condition', 'time', 'batch', 'dnbCount', 'x', 'y', 'area', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'passing_mt', 'passing_ngenes', '_scvi_batch', '_scvi_labels'
     var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
     uns: 'X_tsne', 'X_umap', '_scvi_manager_uuid', '_scvi_uuid', 'hvg', 'log1p', 'neighbors', 'spatial'
     obsm: 'X_s

In [4]:
import numpy as np
import scipy.sparse as sp
import pandas as pd
import gc

# 假设对象叫 sp_adata

# 1. 保留 obs_names 和 var_names
obs_names = sp_adata.obs_names.copy()
var_names = sp_adata.var_names.copy()

# 2. 保留 spatial
if "spatial" not in sp_adata.obsm:
    raise KeyError("sp_adata.obsm 中没有 'spatial'")

spatial = sp_adata.obsm["spatial"].astype(np.float32)

# 3. 保留 X，并压缩 dtype
if sp.issparse(sp_adata.X):
    sp_adata.X = sp_adata.X.tocsr()
    sp_adata.X.data = sp_adata.X.data.astype(np.float32)
else:
    sp_adata.X = sp_adata.X.astype(np.float32)

# 4. 全部清空 obs / var 注释，只保留 index
sp_adata.obs = pd.DataFrame(index=obs_names)
sp_adata.var = pd.DataFrame(index=var_names)

# 5. 只放回 spatial
sp_adata.obsm.clear()
sp_adata.obsm["spatial"] = spatial

# 6. 其他全部清空
sp_adata.layers.clear()
sp_adata.obsp.clear()
sp_adata.varm.clear()
sp_adata.uns.clear()
sp_adata.raw = None

gc.collect()

sp_adata

AnnData object with n_obs × n_vars = 497563 × 19081
    obsm: 'spatial'

In [5]:
sc_adata.write("reference_sc_syg.h5ad")
sp_adata.write("inference_sp.h5ad")

# 注释

In [1]:
%reset -f
import scanpy as sc
from spatialid import Transfer

tr = Transfer(
    single_data="reference_sc_syg.h5ad",
    spatial_data="inference_sp.h5ad",
    output_path="./spatialid_output",
    device=0
)

tr.learn_sc(
    ann_key="celltype",
    marker_genes=None,
    batch_size=384,
    epoch=200
)

# tr.sc2st()

# tr.annotation(
#     pca_dim=200,
#     n_neigh=30,
#     epochs=200
# )

Initializing...
Loading scRNA-seq Data...
Loading spRNA-seq Data...
All data loaded.
Prepare to train...
Training...


1477it [00:13, 106.74it/s]:00<?, ?it/s]

  [2026-05-10 17:05:30] Epoch:   1 Loss: 0.47348, acc: 49.41%


1477it [00:13, 108.99it/s]:15<50:02, 15.09s/it]

  [2026-05-10 17:05:44] Epoch:   2 Loss: 0.40097, acc: 56.32%

...


1477it [00:13, 106.83it/s]49:44<00:15, 15.09s/it]

  [2026-05-10 17:55:14] Epoch: 200 Loss: 0.37910, acc: 57.90%


100%|██████████| 200/200 [49:59<00:00, 15.00s/it]



 validation model: 


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL spatialid.dnn.DNNModel was not an allowed global by default. Please use `torch.serialization.add_safe_globals([spatialid.dnn.DNNModel])` or the `torch.serialization.safe_globals([spatialid.dnn.DNNModel])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types acce